In [1]:
import statsmodels
import torch
import torch.nn as nn
import torch.optim as optim
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
import numpy as np

In [3]:
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Global seed set to {SEED}")

Global seed set to 42


In [4]:
data = pd.read_csv("../data/processed_vol_features.csv", index_col=0, parse_dates=True)

In [5]:
# Compute 30-day realized vol to align with VIX's 30-day implied window
data["realized_vol30"] = data["log_return"].rolling(30).std() * np.sqrt(252)
data = data.dropna().copy()
print(f"Data shape after adding realized_vol30 and dropping NaN: {data.shape}")

Data shape after adding realized_vol30 and dropping NaN: (989, 17)


In [6]:
data.head()

,Close,log_return,realized_vol5,realized_vol10,realized_vol21,vol_of_vol5,parkinson,gk_vol,rv_lag1,rv_lag5,rv_lag21,vix,vix_change,mom_5d,mom_21d,target_rv,realized_vol30
Date,,,,,,,,,,,,,,,,,
2020-03-25,226.859604,0.014859,0.798902,1.123687,0.891781,3.573613,0.513314,0.257705,0.891637,1.423333,0.266841,63.950001,0.036971,0.034338,-0.206009,0.869927,0.743966
2020-03-26,240.105850,0.056749,0.869927,1.043305,0.922072,1.927139,0.527011,0.268806,0.798902,1.265842,0.232205,61.000000,-0.046130,0.092412,-0.156546,0.811764,0.767218
2020-03-27,232.954178,-0.030238,0.811764,0.962499,0.915530,1.358583,0.523377,0.279815,0.869927,1.000785,0.276935,65.540001,0.074426,0.107605,-0.143188,0.699891,0.769169
2020-03-30,240.519470,0.031959,0.699891,0.750531,0.925615,1.188154,0.519818,0.277667,0.811764,0.672446,0.299749,57.080002,-0.129081,0.173581,-0.111630,0.556877,0.777597
2020-03-31,236.934448,-0.015018,0.556877,0.717316,0.909300,1.950112,0.510128,0.274545,0.699891,0.891637,0.534249,53.540001,-0.062018,0.060045,-0.161197,0.687423,0.777544


In [7]:
data.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 989 entries, 2020-03-25 to 2025-12-26
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Close           989 non-null    float64
 1   log_return      989 non-null    float64
 2   realized_vol5   989 non-null    float64
 3   realized_vol10  989 non-null    float64
 4   realized_vol21  989 non-null    float64
 5   vol_of_vol5     989 non-null    float64
 6   parkinson       989 non-null    float64
 7   gk_vol          989 non-null    float64
 8   rv_lag1         989 non-null    float64
 9   rv_lag5         989 non-null    float64
 10  rv_lag21        989 non-null    float64
 11  vix             989 non-null    float64
 12  vix_change      989 non-null    float64
 13  mom_5d          989 non-null    float64
 14  mom_21d         989 non-null    float64
 15  target_rv       989 non-null    float64
 16  realized_vol30  989 non-null    float64
dtypes: float64(17)
memory usage

In [8]:
series = data["realized_vol30"].dropna()

split  = int(len(series) * 0.85)   # 85% train, 15% test

train  = series.iloc[:split]
test   = series.iloc[split:]

print(f"Train: {len(train)} obs  "
      f"({train.index[0].date()} → {train.index[-1].date()})")
print(f"Test : {len(test)}  obs  "
      f"({test.index[0].date()} → {test.index[-1].date()})")

Train: 840 obs  (2020-03-25 → 2024-12-03)
Test : 149  obs  (2024-12-04 → 2025-12-26)


In [9]:
import numpy as np

In [10]:
p, d, q = 2, 0, 1    

history = list(train)
predictions = []

print(f"Running walk-forward forecast "
      f"ARIMA({p},{d},{q}) on {len(test)} steps...")

for i, actual in enumerate(test):
    # Fit on all available history
    model = ARIMA(history, order=(p, d, q))
    fit   = model.fit()

    # Forecast 1 step ahead
    pred  = fit.forecast(steps=1)[0]
    predictions.append(pred)

    # Add actual value to history (walk-forward)
    history.append(actual)

    if i % 50 == 0:
        print(f"  Step {i+1}/{len(test)}  "
              f"pred={pred:.2f}  actual={actual:.2f}")

predictions = np.array(predictions)
actuals  = test.values

Running walk-forward forecast ARIMA(2,0,1) on 149 steps...


/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


  Step 1/149  pred=0.09  actual=0.09


/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Lik

  Step 51/149  pred=0.40  actual=0.40


/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Lik

  Step 101/149  pred=0.09  actual=0.09


/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/josephineamponsah/Documents/price_volatility_prediction/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Lik

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [12]:
#evaluation
mae  = mean_absolute_error(actuals, predictions)
rmse = np.sqrt(mean_squared_error(actuals, predictions))
mape = np.mean(np.abs((actuals - predictions) /
               (actuals + 1e-8))) * 100

# Directional accuracy — did vol go up/down correctly?
actual_dir = np.sign(np.diff(actuals))
pred_dir   = np.sign(np.diff(predictions))
dir_acc    = np.mean(actual_dir == pred_dir) * 100

print(f"  ARIMA({p},{d},{q}) — Baseline Results")
print(f"  MAE             : {mae:.4f}")
print(f"  RMSE            : {rmse:.4f}")
print(f"  MAPE            : {mape:.2f}%")
print(f"  Directional Acc : {dir_acc:.2f}%")

  ARIMA(2,0,1) — Baseline Results
  MAE             : 0.0065
  RMSE            : 0.0196
  MAPE            : 4.20%
  Directional Acc : 53.38%


### Deep Learning Forecasting

In [14]:
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from sklearn.preprocessing import MinMaxScaler

In [15]:
class VolDataset(Dataset):
    def __init__(self, dataframe, feature_cols, target_col, seq_len, horizon = 1, scaler_X = None, scaler_y = None, fit_scalers=False):
        self.df = dataframe
        self.seq_len = seq_len
        self.horizon = horizon
        X_raw = self.df[feature_cols].values.astype(np.float32)
        y_raw = self.df[[target_col]].values.astype(np.float32)
        if fit_scalers:
            self.scaler_X = MinMaxScaler()
            self.scaler_y = MinMaxScaler()
            X_scaled = self.scaler_X.fit_transform(X_raw)
            y_scaled = self.scaler_y.fit_transform(y_raw)
        else:
            self.scaler_X = scaler_X
            self.scaler_y = scaler_y
            X_scaled = self.scaler_X.transform(X_raw)
            y_scaled = self.scaler_y.transform(y_raw)
        self.X, self.y = self._make_sequences(X_scaled, y_scaled)
        
    def _make_sequences(self, X, y):
        X_seq, y_seq = [], []
        total = len(X) - self.seq_len - self.horizon + 1
        for i in range(total):
            X_seq.append(X[i:i+self.seq_len])
            y_seq.append(y[i+self.seq_len+self.horizon-1])
        return (torch.tensor(np.array(X_seq), dtype=torch.float32), torch.tensor(np.array(y_seq), dtype=torch.float32))
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        # row = self.df.iloc[idx]
        # data = row.drop("realized_vol21").values.astype(np.float32)
        # target = row["realized_vol21"].astype(np.float32)
        return self.X[idx], self.y[idx]

In [16]:
target_col   = "realized_vol30"
feature_cols = [col for col in data.columns if col != target_col]
n = len(data)
train_end = int(n * 0.7)
val_end = int(n * 0.85)
train_data = data.iloc[:train_end]
val_data = data.iloc[train_end:val_end]
test_data = data.iloc[val_end:]

In [17]:
train_dataset = VolDataset(train_data, feature_cols, target_col, seq_len=30, horizon=1, fit_scalers=True)
val_dataset = VolDataset(val_data, feature_cols, target_col, seq_len=30, horizon=1, 
                         scaler_X=train_dataset.scaler_X, scaler_y=train_dataset.scaler_y)
test_dataset = VolDataset(test_data, feature_cols, target_col, seq_len=30, horizon=1, 
                          scaler_X=train_dataset.scaler_X, scaler_y=train_dataset.scaler_y)

In [18]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

### MLP Forecaster

In [19]:
class MLPForecaster(nn.Module):
    def __init__(self, seq_len, num_features, hidden_dims, horizon=1, dropout=0.2):
        super().__init__()
        input_dim = seq_len * num_features
        layers = []
        in_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(in_dim, h),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.BatchNorm1d(h)
            ]
            in_dim = h
        layers += [nn.Linear(in_dim, horizon)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten (batch, seq_len*num_features)
        return self.net(x).squeeze()  # (batch, horizon) -> (batch)


In [20]:
model_mlp = MLPForecaster(seq_len=30, num_features=data.shape[1]-1, hidden_dims=[128, 64], horizon=1, dropout=0.2)

In [21]:
def train_model(model, train_loader, val_loader, epochs=20, lr=1e-3,
                optimizer_name="adam", criterion_name="mse",
                use_scheduler=False, clip_grad=None, model_name="model"):
    """
    Unified training loop.
    
    Args:
        optimizer_name : "adam" | "adamw" | "rmsprop"
        criterion_name : "mse"  | "huber"
        use_scheduler  : ReduceLROnPlateau if True
        clip_grad      : max grad norm (None = disabled)
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.HuberLoss() if criterion_name == "huber" else nn.MSELoss()

    if optimizer_name == "adamw":
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    elif optimizer_name == "rmsprop":
        optimizer = optim.RMSprop(model.parameters(), lr=lr)
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr)

    scheduler = (optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
                 if use_scheduler else None)

    best_val_loss = float("inf")
    for epoch in range(epochs):
        model.train()
        train_losses = []
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b.squeeze())
            loss.backward()
            if clip_grad:
                nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_v, y_v in val_loader:
                X_v, y_v = X_v.to(device), y_v.to(device)
                val_losses.append(criterion(model(X_v), y_v.squeeze()).item())

        avg_train = np.mean(train_losses)
        avg_val   = np.mean(val_losses)

        if scheduler:
            scheduler.step(avg_val)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), f"best_{model_name}.pth")

        current_lr = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch+1}/{epochs}  Train: {avg_train:.4f}  Val: {avg_val:.4f}  LR: {current_lr:.2e}")

    model.load_state_dict(torch.load(f"best_{model_name}.pth"))
    return model

In [22]:

trained_mlp = train_model(model_mlp, train_loader, val_loader, epochs=20, lr=1e-3, model_name="MLPForecaster")

Epoch 1/20  Train: 0.1736  Val: 0.0182  LR: 1.00e-03
Epoch 2/20  Train: 0.1036  Val: 0.0151  LR: 1.00e-03
Epoch 3/20  Train: 0.0744  Val: 0.0044  LR: 1.00e-03
Epoch 4/20  Train: 0.0623  Val: 0.0055  LR: 1.00e-03
Epoch 5/20  Train: 0.0570  Val: 0.0017  LR: 1.00e-03
Epoch 6/20  Train: 0.0389  Val: 0.0025  LR: 1.00e-03
Epoch 7/20  Train: 0.0324  Val: 0.0082  LR: 1.00e-03
Epoch 8/20  Train: 0.0303  Val: 0.0087  LR: 1.00e-03
Epoch 9/20  Train: 0.0245  Val: 0.0157  LR: 1.00e-03
Epoch 10/20  Train: 0.0208  Val: 0.0255  LR: 1.00e-03
Epoch 11/20  Train: 0.0156  Val: 0.0374  LR: 1.00e-03
Epoch 12/20  Train: 0.0156  Val: 0.0367  LR: 1.00e-03
Epoch 13/20  Train: 0.0142  Val: 0.0271  LR: 1.00e-03
Epoch 14/20  Train: 0.0104  Val: 0.0253  LR: 1.00e-03
Epoch 15/20  Train: 0.0098  Val: 0.0209  LR: 1.00e-03
Epoch 16/20  Train: 0.0088  Val: 0.0152  LR: 1.00e-03
Epoch 17/20  Train: 0.0066  Val: 0.0135  LR: 1.00e-03
Epoch 18/20  Train: 0.0070  Val: 0.0134  LR: 1.00e-03
Epoch 19/20  Train: 0.0062  Val: 0.01

In [23]:
def evaluate_model(model, test_loader, scaler_y, model_name="Model"):
    model.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        preds, actuals = [], []
        for X_test, y_test in test_loader:
            X_test, y_test = X_test.to(device), y_test.to(device)
            preds.append(model(X_test).cpu().numpy())
            actuals.append(y_test.cpu().numpy())
    preds   = np.concatenate(preds,   axis=0)
    actuals = np.concatenate(actuals, axis=0)
    preds_orig   = scaler_y.inverse_transform(preds.reshape(-1, 1)).flatten()
    actuals_orig = scaler_y.inverse_transform(actuals.reshape(-1, 1)).flatten()
    return _print_metrics(actuals_orig, preds_orig, model_name)


def _print_metrics(actuals, preds, model_name):
    mae      = mean_absolute_error(actuals, preds)
    rmse     = np.sqrt(mean_squared_error(actuals, preds))
    mape     = np.mean(np.abs((actuals - preds) / (actuals + 1e-8))) * 100
    dir_acc  = np.mean(np.sign(np.diff(actuals)) == np.sign(np.diff(preds))) * 100
    print(f"{model_name}")
    print(f"  MAE             : {mae:.4f}")
    print(f"  RMSE            : {rmse:.4f}")
    print(f"  MAPE            : {mape:.2f}%")
    print(f"  Directional Acc : {dir_acc:.2f}%")
    return {"Model": model_name, "MAE": mae, "RMSE": rmse, "MAPE": mape, "Dir Acc": dir_acc}

In [24]:
evaluate_model(trained_mlp, test_loader, train_dataset.scaler_y, model_name="MLP Forecaster")

MLP Forecaster
  MAE             : 0.0916
  RMSE            : 0.1107
  MAPE            : 60.76%
  Directional Acc : 48.31%


{'Model': 'MLP Forecaster',
 'MAE': 0.09160330891609192,
 'RMSE': np.float64(0.1107361766657315),
 'MAPE': np.float32(60.76125),
 'Dir Acc': np.float64(48.30508474576271)}

### LSTM Forecaster

In [25]:
class LSTMForecaster(nn.Module):
    """
    Stacked LSTM forecaster.
    Input:  (batch, seq_len, n_features)
    Output: (batch,) when horizon=1, else (batch, horizon)
    """
    def __init__(self, input_size=1, hidden_size=64,
                 num_layers=2, horizon=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,          # (batch, seq, features)
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # Use final hidden state from last layer
        out = self.dropout(h_n[-1])   # (batch, hidden_size)
        return self.fc(out).squeeze(-1)  # (batch, horizon) -> (batch,) when horizon=1


In [26]:
lstm_model = LSTMForecaster(input_size=data.shape[1]-1, hidden_size=64, num_layers=2)

In [27]:
trained_lstm = train_model(lstm_model, train_loader, val_loader, epochs=20, lr=1e-3, model_name="LSTMForecaster")

Epoch 1/20  Train: 0.0057  Val: 0.0017  LR: 1.00e-03
Epoch 2/20  Train: 0.0028  Val: 0.0089  LR: 1.00e-03
Epoch 3/20  Train: 0.0019  Val: 0.0038  LR: 1.00e-03
Epoch 4/20  Train: 0.0016  Val: 0.0050  LR: 1.00e-03
Epoch 5/20  Train: 0.0012  Val: 0.0035  LR: 1.00e-03
Epoch 6/20  Train: 0.0009  Val: 0.0019  LR: 1.00e-03
Epoch 7/20  Train: 0.0008  Val: 0.0009  LR: 1.00e-03
Epoch 8/20  Train: 0.0007  Val: 0.0004  LR: 1.00e-03
Epoch 9/20  Train: 0.0006  Val: 0.0009  LR: 1.00e-03
Epoch 10/20  Train: 0.0005  Val: 0.0005  LR: 1.00e-03
Epoch 11/20  Train: 0.0005  Val: 0.0004  LR: 1.00e-03
Epoch 12/20  Train: 0.0005  Val: 0.0005  LR: 1.00e-03
Epoch 13/20  Train: 0.0004  Val: 0.0005  LR: 1.00e-03
Epoch 14/20  Train: 0.0004  Val: 0.0005  LR: 1.00e-03
Epoch 15/20  Train: 0.0004  Val: 0.0004  LR: 1.00e-03
Epoch 16/20  Train: 0.0004  Val: 0.0003  LR: 1.00e-03
Epoch 17/20  Train: 0.0004  Val: 0.0002  LR: 1.00e-03
Epoch 18/20  Train: 0.0004  Val: 0.0003  LR: 1.00e-03
Epoch 19/20  Train: 0.0003  Val: 0.00

In [28]:
evaluate_model(trained_lstm, test_loader, train_dataset.scaler_y, model_name="LSTM Forecaster")

LSTM Forecaster
  MAE             : 0.0304
  RMSE            : 0.0458
  MAPE            : 14.51%
  Directional Acc : 53.39%


{'Model': 'LSTM Forecaster',
 'MAE': 0.030364206060767174,
 'RMSE': np.float64(0.04578841241672193),
 'MAPE': np.float32(14.511758),
 'Dir Acc': np.float64(53.38983050847458)}

In [29]:
### LSTM Hyperparameter Search
#Grid search over learning rates, epoch counts, and optimizers to find the best configuration.

In [30]:
import itertools

def lstm_hparam_search(train_loader, val_loader, input_size,
                       lr_list, epoch_list, optimizer_list):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results = []

    for lr, epochs, opt_name in itertools.product(lr_list, epoch_list, optimizer_list):
        model = LSTMForecaster(input_size=input_size, hidden_size=64, num_layers=2)
        model.to(device)

        if opt_name == "adamw":
            optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        elif opt_name == "rmsprop":
            optimizer = optim.RMSprop(model.parameters(), lr=lr)
        else:
            optimizer = optim.Adam(model.parameters(), lr=lr)

        criterion = nn.MSELoss()
        best_val = float("inf")

        for _ in range(epochs):
            model.train()
            for X_b, y_b in train_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                optimizer.zero_grad()
                loss = criterion(model(X_b), y_b.squeeze())
                loss.backward()
                optimizer.step()

            model.eval()
            val_losses = []
            with torch.no_grad():
                for X_v, y_v in val_loader:
                    X_v, y_v = X_v.to(device), y_v.to(device)
                    val_losses.append(criterion(model(X_v), y_v.squeeze()).item())
            best_val = min(best_val, np.mean(val_losses))

        results.append({"lr": lr, "epochs": epochs, "optimizer": opt_name, "best_val_loss": best_val})
        print(f"  lr={lr:.0e}  epochs={epochs:3d}  opt={opt_name:8s}  best_val_loss={best_val:.6f}")

    results_df = (pd.DataFrame(results)
                  .sort_values("best_val_loss")
                  .reset_index(drop=True))
    print("\nTop 5 configs:")
    print(results_df.head())
    return results_df

search_results = lstm_hparam_search(
    train_loader, val_loader,
    input_size=data.shape[1] - 1,
    lr_list=[1e-2, 1e-3, 5e-4],
    epoch_list=[20, 40],
    optimizer_list=["adam", "adamw", "rmsprop"],
)

  lr=1e-02  epochs= 20  opt=adam      best_val_loss=0.000390
  lr=1e-02  epochs= 20  opt=adamw     best_val_loss=0.000579
  lr=1e-02  epochs= 20  opt=rmsprop   best_val_loss=0.000769
  lr=1e-02  epochs= 40  opt=adam      best_val_loss=0.000401
  lr=1e-02  epochs= 40  opt=adamw     best_val_loss=0.000349
  lr=1e-02  epochs= 40  opt=rmsprop   best_val_loss=0.000578
  lr=1e-03  epochs= 20  opt=adam      best_val_loss=0.000191
  lr=1e-03  epochs= 20  opt=adamw     best_val_loss=0.000254
  lr=1e-03  epochs= 20  opt=rmsprop   best_val_loss=0.000593
  lr=1e-03  epochs= 40  opt=adam      best_val_loss=0.000189
  lr=1e-03  epochs= 40  opt=adamw     best_val_loss=0.000243
  lr=1e-03  epochs= 40  opt=rmsprop   best_val_loss=0.000332
  lr=5e-04  epochs= 20  opt=adam      best_val_loss=0.000476
  lr=5e-04  epochs= 20  opt=adamw     best_val_loss=0.000547
  lr=5e-04  epochs= 20  opt=rmsprop   best_val_loss=0.000479
  lr=5e-04  epochs= 40  opt=adam      best_val_loss=0.000237
  lr=5e-04  epochs= 40  

### LSTM V2 — Best Config with Optimizer & LR Scheduler
Uses the best (lr, epochs, optimizer) found above, plus `ReduceLROnPlateau` and gradient clipping.

In [31]:
# train_model_v2 merged into train_model above.
# Equivalent call: train_model(..., optimizer_name="adamw", use_scheduler=True, clip_grad=1.0)

In [32]:
best = search_results.iloc[0]
print(f"Best config — lr={best.lr:.0e}  epochs={int(best.epochs)}  optimizer={best.optimizer}  val_loss={best.best_val_loss:.6f}")

lstm_v2 = LSTMForecaster(input_size=data.shape[1] - 1, hidden_size=128, num_layers=2)
trained_lstm_v2 = train_model(
    lstm_v2, train_loader, val_loader,
    epochs=int(best.epochs),
    lr=best.lr,
    optimizer_name=best.optimizer,
    use_scheduler=True,
    clip_grad=1.0,
    model_name="LSTMForecaster_v2",
)

Best config — lr=5e-04  epochs=40  optimizer=adamw  val_loss=0.000180
Epoch 1/40  Train: 0.0114  Val: 0.0099  LR: 5.00e-04
Epoch 2/40  Train: 0.0060  Val: 0.0021  LR: 5.00e-04
Epoch 3/40  Train: 0.0048  Val: 0.0025  LR: 5.00e-04
Epoch 4/40  Train: 0.0034  Val: 0.0013  LR: 5.00e-04
Epoch 5/40  Train: 0.0016  Val: 0.0050  LR: 5.00e-04
Epoch 6/40  Train: 0.0010  Val: 0.0030  LR: 5.00e-04
Epoch 7/40  Train: 0.0009  Val: 0.0035  LR: 5.00e-04
Epoch 8/40  Train: 0.0009  Val: 0.0024  LR: 5.00e-04
Epoch 9/40  Train: 0.0008  Val: 0.0015  LR: 5.00e-04
Epoch 10/40  Train: 0.0006  Val: 0.0017  LR: 2.50e-04
Epoch 11/40  Train: 0.0005  Val: 0.0016  LR: 2.50e-04
Epoch 12/40  Train: 0.0005  Val: 0.0011  LR: 2.50e-04
Epoch 13/40  Train: 0.0005  Val: 0.0012  LR: 2.50e-04
Epoch 14/40  Train: 0.0005  Val: 0.0009  LR: 2.50e-04
Epoch 15/40  Train: 0.0005  Val: 0.0009  LR: 2.50e-04
Epoch 16/40  Train: 0.0005  Val: 0.0006  LR: 2.50e-04
Epoch 17/40  Train: 0.0004  Val: 0.0006  LR: 2.50e-04
Epoch 18/40  Train: 0

In [33]:
evaluate_model(trained_lstm_v2, test_loader, train_dataset.scaler_y, model_name="LSTM V2 (Tuned)")

LSTM V2 (Tuned)
  MAE             : 0.0244
  RMSE            : 0.0410
  MAPE            : 12.01%
  Directional Acc : 53.39%


{'Model': 'LSTM V2 (Tuned)',
 'MAE': 0.02440015599131584,
 'RMSE': np.float64(0.04097244851158534),
 'MAPE': np.float32(12.007221),
 'Dir Acc': np.float64(53.38983050847458)}

### LSTM V3 — HuberLoss + Larger Hidden + Higher Dropout
`hidden_size=256`, `num_layers=3`, `dropout=0.3`, `criterion=HuberLoss`

In [34]:
lstm_v3 = LSTMForecaster(input_size=data.shape[1] - 1, hidden_size=256, num_layers=3, dropout=0.3)
trained_lstm_v3 = train_model(
    lstm_v3, train_loader, val_loader,
    epochs=int(best.epochs),
    lr=best.lr,
    optimizer_name=best.optimizer,
    criterion_name="huber",
    use_scheduler=True,
    clip_grad=1.0,
    model_name="LSTMForecaster_v3",
)

Epoch 1/40  Train: 0.0044  Val: 0.0012  LR: 5.00e-04
Epoch 2/40  Train: 0.0027  Val: 0.0012  LR: 5.00e-04
Epoch 3/40  Train: 0.0009  Val: 0.0021  LR: 5.00e-04
Epoch 4/40  Train: 0.0004  Val: 0.0017  LR: 5.00e-04
Epoch 5/40  Train: 0.0004  Val: 0.0004  LR: 5.00e-04
Epoch 6/40  Train: 0.0003  Val: 0.0002  LR: 5.00e-04
Epoch 7/40  Train: 0.0002  Val: 0.0001  LR: 5.00e-04
Epoch 8/40  Train: 0.0002  Val: 0.0001  LR: 5.00e-04
Epoch 9/40  Train: 0.0002  Val: 0.0001  LR: 5.00e-04
Epoch 10/40  Train: 0.0002  Val: 0.0001  LR: 5.00e-04
Epoch 11/40  Train: 0.0002  Val: 0.0002  LR: 5.00e-04
Epoch 12/40  Train: 0.0002  Val: 0.0001  LR: 5.00e-04
Epoch 13/40  Train: 0.0002  Val: 0.0001  LR: 5.00e-04
Epoch 14/40  Train: 0.0002  Val: 0.0001  LR: 5.00e-04
Epoch 15/40  Train: 0.0002  Val: 0.0001  LR: 5.00e-04
Epoch 16/40  Train: 0.0002  Val: 0.0001  LR: 5.00e-04
Epoch 17/40  Train: 0.0002  Val: 0.0001  LR: 5.00e-04
Epoch 18/40  Train: 0.0002  Val: 0.0001  LR: 5.00e-04
Epoch 19/40  Train: 0.0002  Val: 0.00

In [35]:
evaluate_model(trained_lstm_v3, test_loader, train_dataset.scaler_y, model_name="LSTM V3 (Huber + Deeper)")

LSTM V3 (Huber + Deeper)
  MAE             : 0.0318
  RMSE            : 0.0447
  MAPE            : 18.13%
  Directional Acc : 53.39%


{'Model': 'LSTM V3 (Huber + Deeper)',
 'MAE': 0.031757958233356476,
 'RMSE': np.float64(0.04469433981395973),
 'MAPE': np.float32(18.129917),
 'Dir Acc': np.float64(53.38983050847458)}

### CNN-LSTM Forecaster
1D-CNN extracts local feature patterns across the time axis; LSTM captures long-range temporal dependencies.

In [36]:
class CNNLSTMForecaster(nn.Module):
    """
    CNN-LSTM hybrid.
      - Two stacked Conv1d layers extract short-range local patterns.
      - LSTM processes the CNN feature map to capture long-range dependencies.
    Input:  (batch, seq_len, n_features)
    Output: (batch,) when horizon=1, else (batch, horizon)
    """
    def __init__(self, input_size, num_filters=64, kernel_size=3,
                 hidden_size=128, num_layers=2, horizon=1, dropout=0.2):
        super().__init__()
        self.cnn = nn.Sequential(
            # Conv1d expects (batch, channels, seq_len); channels = n_features
            nn.Conv1d(input_size, num_filters, kernel_size, padding=kernel_size // 2),
            nn.ReLU(),
            nn.Conv1d(num_filters, num_filters, kernel_size, padding=kernel_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(
            input_size=num_filters,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        x = x.permute(0, 2, 1)           # → (batch, n_features, seq_len)
        x = self.cnn(x)                  # → (batch, num_filters, seq_len)
        x = x.permute(0, 2, 1)           # → (batch, seq_len, num_filters)
        _, (h_n, _) = self.lstm(x)
        out = self.dropout(h_n[-1])      # (batch, hidden_size)
        return self.fc(out).squeeze(-1)  # (batch,)

In [37]:
cnn_lstm_model = CNNLSTMForecaster(
    input_size=data.shape[1] - 1,
    num_filters=64,
    kernel_size=3,
    hidden_size=128,
    num_layers=2,
    horizon=1,
    dropout=0.2,
)

In [38]:
trained_cnn_lstm = train_model(
    cnn_lstm_model, train_loader, val_loader,
    epochs=int(best.epochs),
    lr=best.lr,
    optimizer_name=best.optimizer,
    use_scheduler=True,
    clip_grad=1.0,
    model_name="CNNLSTMForecaster",
)

Epoch 1/40  Train: 0.0066  Val: 0.0027  LR: 5.00e-04
Epoch 2/40  Train: 0.0056  Val: 0.0037  LR: 5.00e-04
Epoch 3/40  Train: 0.0023  Val: 0.0030  LR: 5.00e-04
Epoch 4/40  Train: 0.0016  Val: 0.0010  LR: 5.00e-04
Epoch 5/40  Train: 0.0013  Val: 0.0011  LR: 5.00e-04
Epoch 6/40  Train: 0.0008  Val: 0.0016  LR: 5.00e-04
Epoch 7/40  Train: 0.0007  Val: 0.0011  LR: 5.00e-04
Epoch 8/40  Train: 0.0006  Val: 0.0012  LR: 5.00e-04
Epoch 9/40  Train: 0.0005  Val: 0.0013  LR: 5.00e-04
Epoch 10/40  Train: 0.0005  Val: 0.0009  LR: 5.00e-04
Epoch 11/40  Train: 0.0005  Val: 0.0014  LR: 5.00e-04
Epoch 12/40  Train: 0.0005  Val: 0.0010  LR: 5.00e-04
Epoch 13/40  Train: 0.0005  Val: 0.0004  LR: 5.00e-04
Epoch 14/40  Train: 0.0005  Val: 0.0008  LR: 5.00e-04
Epoch 15/40  Train: 0.0004  Val: 0.0007  LR: 5.00e-04
Epoch 16/40  Train: 0.0005  Val: 0.0005  LR: 5.00e-04
Epoch 17/40  Train: 0.0004  Val: 0.0006  LR: 5.00e-04
Epoch 18/40  Train: 0.0004  Val: 0.0007  LR: 5.00e-04
Epoch 19/40  Train: 0.0004  Val: 0.00

In [39]:
evaluate_model(trained_cnn_lstm, test_loader, train_dataset.scaler_y, model_name="CNN-LSTM Forecaster")

CNN-LSTM Forecaster
  MAE             : 0.0369
  RMSE            : 0.0455
  MAPE            : 26.14%
  Directional Acc : 55.93%


{'Model': 'CNN-LSTM Forecaster',
 'MAE': 0.036923572421073914,
 'RMSE': np.float64(0.04553990900104007),
 'MAPE': np.float32(26.137817),
 'Dir Acc': np.float64(55.932203389830505)}

---
## Model Comparison vs Market (IV) Baseline
**Alignment:** VIX on day `t-30` is the market's forecast for the 30-day window `[t-30 → t]`.
`realized_vol30` on day `t` covers exactly that same window. So the correct pairing is:

```
VIX(t-30) / 100  →  compared against  →  realized_vol30(t)
```

This is implemented via `shift(30)` on the VIX series. The first 30 rows of the IV series become NaN and are dropped, so the IV baseline covers a slightly shorter window than the trained models (30 fewer observations at the start of the test set).

In [41]:
# ── Comparison table ──────────────────────────────────────────────────────────
results = []

# IV market baseline — proper temporal alignment
# VIX(t-30) was the market forecast for the window ending at t;
# realized_vol30(t) is what actually happened over that window.
seq_len = train_dataset.seq_len

iv_shifted = test_data["vix"].shift(30) / 100          # VIX from 30 days prior, % → decimal
alignment  = pd.concat(
    [iv_shifted.rename("iv_pred"), test_data["realized_vol30"].rename("rv_actual")],
    axis=1
).iloc[seq_len:].dropna()                               # clip to model window, drop NaN lead

iv_preds_arr   = alignment["iv_pred"].values
rv_actuals_arr = alignment["rv_actual"].values

print(f"IV baseline sample size : {len(iv_preds_arr)} "
      f"(model test window : {len(test_dataset)} samples)")
results.append(_print_metrics(rv_actuals_arr, iv_preds_arr, "IV Baseline (VIX shifted 30d)"))
print()

# All trained models (fast inference only — no re-training)
for name, mdl in [
    ("MLP",          trained_mlp),
    ("LSTM V1",      trained_lstm),
    ("LSTM V2",      trained_lstm_v2),
    ("LSTM V3",      trained_lstm_v3),
    ("CNN-LSTM",     trained_cnn_lstm),
]:
    results.append(evaluate_model(mdl, test_loader, train_dataset.scaler_y, model_name=name))
    print()

summary = (pd.DataFrame(results)
             .set_index("Model")
             .sort_values("MAPE"))
summary["MAPE"]    = summary["MAPE"].map("{:.2f}%".format)
summary["Dir Acc"] = summary["Dir Acc"].map("{:.2f}%".format)
summary["MAE"]     = summary["MAE"].map("{:.4f}".format)
summary["RMSE"]    = summary["RMSE"].map("{:.4f}".format)

print("\n── Summary (sorted by MAPE) ──────────────────────────────")
print(summary.to_string())

IV baseline sample size : 119 (model test window : 119 samples)
IV Baseline (VIX shifted 30d)
  MAE             : 0.1271
  RMSE            : 0.1416
  MAPE            : 90.09%
  Directional Acc : 46.61%

MLP
  MAE             : 0.0916
  RMSE            : 0.1107
  MAPE            : 60.76%
  Directional Acc : 48.31%

LSTM V1
  MAE             : 0.0304
  RMSE            : 0.0458
  MAPE            : 14.51%
  Directional Acc : 53.39%

LSTM V2
  MAE             : 0.0244
  RMSE            : 0.0410
  MAPE            : 12.01%
  Directional Acc : 53.39%

LSTM V3
  MAE             : 0.0318
  RMSE            : 0.0447
  MAPE            : 18.13%
  Directional Acc : 53.39%

CNN-LSTM
  MAE             : 0.0369
  RMSE            : 0.0455
  MAPE            : 26.14%
  Directional Acc : 55.93%


── Summary (sorted by MAPE) ──────────────────────────────
                                  MAE    RMSE    MAPE Dir Acc
Model                                                        
LSTM V2                        